# demo_llm

> Phase 10: language-model agents (Claude API) on text foraging and the task board, with the Phase 7 channels; the model-size contrast, tokens as compute, numpy versus LLM, and the prompt-only arms.

Spec: `specs/multiagent/phase10/` (design-low-level section 9). Artifacts under `results/multiagent/llm/{quick,pilot,full}/` (gitignored): `runs.csv`, `turns.jsonl`, `metrics.json`, `report.md` and figures; the response cache and the spend ledger live in `results/multiagent/llm/`.

- **Quick** (normal tests): every grid and both runners at 3 turns with the offline `FakeClient`, a pipeline check. No network, no `anthropic` package, no credentials.
- **Pilot** (`--flags llm_pilot`) and **main grids** (`--cell-timeout 0 --flags llm`) make live API calls, and only when `EXPLRL_LLM_LIVE=1` is set; every request is capped by the spending cap below and cached, so a rerun is free and exact.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import time

import numpy as np

from explRL.bench_grid import runGrid, writeCSV
from explRL.llm_client import PRICES, Budget, BudgetExceeded
from explRL.llm_figures import writeFigures
from explRL.llm_grid import (ANALYSIS_PLAN, GRIDS, SETTINGS, estimateGrid, numpyGrid, pilotGrid, planHash, quickGrid,
                             runAnyCell, withClient)
from explRL.llm_report import summaries, writeReport
from explRL.train import demo_artifact_dir

DN = 'multiagent/llm'
WORKERS = min(8, os.cpu_count() or 1)
CAP = 150.0                                    # dollars; decided by the user 2026-10-11 (decision L2)
ROOT = demo_artifact_dir(None, DN)
LIVE = os.environ.get('EXPLRL_LLM_LIVE') == '1'
LIVE_SPEC = dict(kind='live', cache=str(ROOT / 'cache'), cap=CAP, ledger=str(ROOT / 'spend.jsonl'))


def gitCommit():
    try:
        return subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'unknown'
    except OSError:
        return 'unknown'


def _keyed(d):
    return {('|'.join(map(str, k)) if isinstance(k, tuple) else k): _keyed(v) if isinstance(v, dict) else v
            for k, v in d.items()}


def run(mode, grids, spec, workers=WORKERS):
    """Run named grids in order (stopping at the cap), then write runs.csv, turns.jsonl, figures, metrics.json and
    report.md under results/multiagent/llm/<mode>/."""
    out = demo_artifact_dir(mode, DN)
    logs = out / 'turns'
    shutil.rmtree(logs, ignore_errors=True)
    nLive = lambda: sum(1 for _ in open(spec['ledger'])) if spec.get('ledger') and os.path.exists(spec['ledger']) else 0
    rows, done, stopped, t0, live0 = [], [], None, time.time(), nLive()
    for name, cells in grids:
        try:
            rows += runGrid(withClient(cells, spec, str(logs)), workers, run=runAnyCell)
            done.append(name)
        except BudgetExceeded as e:
            stopped = f'{name}: {e}'
            break
    wall, live = time.time() - t0, nLive() - live0
    writeCSV(rows, out / 'runs.csv')
    with open(out / 'turns.jsonl', 'w') as f:
        for p in sorted(logs.glob('*.jsonl')) if logs.exists() else []:
            f.write(p.read_text())
    shutil.rmtree(logs, ignore_errors=True)
    llm, npy = [r for r in rows if r['runner'] == 'llm'], [r for r in rows if r['runner'] == 'numpy']
    S = summaries(llm, npy)
    models = sorted({r['model'] for r in llm})
    meta = dict(mode=mode, planHash=planHash(), commit=gitCommit(), models=models,
                effort=', '.join(sorted({r['effort'] for r in llm})), prices={m: PRICES[m] for m in models},
                pricesChecked=', '.join(sorted({PRICES[m]['checked'] for m in models})), cap=CAP,
                cost=float(sum(r['cost'] for r in llm)), calls=int(sum(r['calls'] for r in llm)),
                wall=wall)
    meta['cacheHits'] = meta['calls'] - live if spec.get('ledger') else 0     # served from the response cache
    writeReport(out / 'report.md', S, meta)
    writeFigures(out, llm, npy, S)
    spent = Budget(CAP, spec['ledger']).spent() if spec.get('ledger') else 0.0
    est = {name: estimateGrid(cells) for name, cells in grids}
    (out / 'metrics.json').write_text(json.dumps(dict(meta, grids=[n for n, _ in grids], done=done, stopped=stopped,
                                                      spentTotal=spent, plan=ANALYSIS_PLAN, estimate=est,
                                                      summaries=_keyed(S)), indent=2, sort_keys=True, default=str) + '\n')
    print(f"{mode}: {len(rows)} rows, {meta['calls']} calls ({meta['cacheHits']} from the cache), "
          f"${meta['cost']:.2f} for these runs, ${spent:.2f} spent in all, {wall:.0f} s; stopped: {stopped}")
    return rows, S, meta

## Quick: every grid and both runners with the fake client (pipeline check)

The estimate printed first is for the full grids, from the provisional token profile; the pilot replaces the profile.

In [ ]:
for name, g in GRIDS.items():
    e = estimateGrid(g())
    print(f'{name:8s}', {m: (v['calls'] + v['beliefCalls'], round(v['cost'], 2)) for m, v in e.items()})
print('pilot   ', estimateGrid(pilotGrid(1) + pilotGrid(2, SETTINGS['difficulty'])))
t0 = time.time()
rows, S, meta = run('quick', [('quick', quickGrid())], dict(kind='fake'))
assert time.time() - t0 < 30
q = demo_artifact_dir('quick', DN)
for f in ('runs.csv', 'turns.jsonl', 'metrics.json', 'report.md', 'welfare.png', 'model_size.png', 'tokens.png',
          'numpy_vs_llm.png', 'arms.png'):
    assert (q / f).stat().st_size > 0, f
# the CSV does not depend on the number of workers
csv4 = (q / 'runs.csv').read_bytes()
run('quick', [('quick', quickGrid())], dict(kind='fake'), workers=1)
assert (q / 'runs.csv').read_bytes() == csv4
# nothing key-like in any output
for p in q.iterdir():
    if p.suffix in ('.csv', '.jsonl', '.json', '.md'):
        assert not re.search(r'sk-ant-|x-api-key|authorization|api[_-]?key', p.read_text(), re.I), p
print(open(q / 'report.md').read()[:1500])

## Pilot (`--flags llm_pilot`, live, about 1 000 calls)

Stage 1: the single-agent task board at difficulties 1-4 (Claude Haiku 5.5, effort low, seeds 1000-1004) and text foraging; the difficulty with a solve rate (tasks solved per task opened) in 30-70%, closest to 50%, is chosen. Stage 2 at that difficulty: effort medium, and Claude Sonnet 5.5. The rerun of stage 1 inside stage 2 is served from the cache. The choices, the measured token profile and the invalid-action rate are printed for the README (task T7).

In [ ]:
#| llm_pilot
if not LIVE:
    print('skipped: set EXPLRL_LLM_LIVE=1 to make live API calls')
else:
    rows1, _, _ = run('pilot', [('pilot1', pilotGrid(1))], LIVE_SPEC, workers=4)
    rate = {}
    for r in rows1:
        if r['task'] == 'board':
            rate.setdefault(int(r['difficulty']), []).append(r['solved'] / max(r['opens'], 1))
    rate = {d: float(np.mean(v)) for d, v in sorted(rate.items())}
    ok = {d: v for d, v in rate.items() if 0.3 <= v <= 0.7}
    choice = min(ok or rate, key=lambda d: abs(rate[d] - 0.5))
    print('solve rate by difficulty:', rate, '-> difficulty', choice)
    rows, S, meta = run('pilot', [('pilot1', pilotGrid(1)), ('pilot2', pilotGrid(2, choice))], LIVE_SPEC, workers=4)
    llm = [r for r in rows if r['runner'] == 'llm']
    prof = {}
    for r in llm:
        prof.setdefault((r['model'], r['task']), []).append(r)
    for k, rs in sorted(prof.items()):
        c = sum(r['calls'] for r in rs)
        print(k, 'per call: uncached input', round(sum(r['inputTokens'] + r['cacheWriteTokens'] for r in rs) / c),
              'cache read', round(sum(r['cacheReadTokens'] for r in rs) / c), 'output',
              round(sum(r['outputTokens'] for r in rs) / c), '| invalid', round(sum(r['invalid'] for r in rs) / c, 3),
              '| max_tokens stops', sum(r['maxTokenStops'] for r in rs))
    for lab, sel in (('effort medium', lambda r: r['level'] == 'board|medium'),
                     ('sonnet', lambda r: r['level'] == 'board|sonnet'),
                     ('haiku low at choice', lambda r: r['level'] == f'board|d={choice}')):
        rs = [r for r in llm if sel(r)]
        print(lab, 'solve rate', round(float(np.mean([r['solved'] / max(r['opens'], 1) for r in rs])), 3) if rs else None)

## Main grids and arms (`--cell-timeout 0 --flags llm`, live)

Grids run in priority order (main, Sonnet, equal budget, liar, framing, memory, persistence) so that a cap reached late costs the least important arms; the numpy teams run on the main cells. Settings come from the pilot (`SETTINGS`), and the analysis plan must be frozen first.

In [ ]:
#| llm
if not LIVE:
    print('skipped: set EXPLRL_LLM_LIVE=1 to make live API calls')
else:
    assert ANALYSIS_PLAN['frozen'], 'freeze the analysis plan (task T7) before the main runs'
    grids = [(name, g()) for name, g in GRIDS.items()] + [('numpy', numpyGrid('forage') + numpyGrid('board'))]
    rows, S, meta = run('full', grids, LIVE_SPEC)